# Production RAG Pipeline

Day 5 of Week 1. The minimal RAG pipeline from [notebook 03](/courses/llm-eng/03-rag-concepts.html) used numpy for retrieval and a hand-written corpus. This notebook upgrades every component to production quality: ChromaDB as the vector store (with persistent storage and metadata filtering), real SEC-style filings as the corpus, hybrid retrieval combining dense similarity with BM25 keyword matching fused via Reciprocal Rank Fusion, and cross-encoder reranking as a final precision pass. We evaluate the complete pipeline with RAGAS and show quantitatively how each upgrade improves Recall@$k$.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## What Changes at Production Scale

The numpy pipeline from [notebook 03](/courses/llm-eng/03-rag-concepts.html) is correct but has three gaps that matter at production scale.

**Scale.** numpy's brute-force cosine search is $O(n \cdot d)$ per query. At 100k chunks and $d = 1536$, that is 153M floating-point multiplications per query — acceptable in a notebook, fatal in a service handling hundreds of concurrent requests. ChromaDB uses HNSW (Hierarchical Navigable Small World) indexing, which reduces query complexity to approximately $O(\log n)$ with a small recall tradeoff.

<br>

**Exact-match requirements.** Financial corpora are dense with identifiers that have no semantic meaning: CUSIPs, LEIs, ISIN codes, regulation numbers ("Reg T", "Basel III", "17 CFR 240.15c3-1"). A dense embedding model collapses these identifiers into vector directions determined by their surrounding context — it may map "CET1" and "Common Equity Tier 1" to similar vectors (good) but may also fail to distinguish two similar regulatory codes (bad). BM25 term-frequency matching finds exact identifiers that dense retrieval misses.

<br>

**ANN approximation error.** Approximate Nearest Neighbor search trades a small amount of recall for large speed gains. A cross-encoder reranker closes this gap: it takes the top-20 ANN results and re-scores each one with a full bidirectional attention pass over the (query, passage) pair, which is far more accurate than an asymmetric dot product but too expensive to run on the full corpus.

## ChromaDB Vector Store

We wrap `chromadb.PersistentClient` in a `ChromaStore` class that exposes the same interface as the numpy `VectorStore` from notebook 03. The key addition is metadata filtering: we tag each chunk with its section name and can restrict retrieval to a specific section using ChromaDB's `where` parameter.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction


class ChromaStore:
    """Persistent ChromaDB vector store with metadata-filter support."""

    def __init__(self, persist_dir: str = "/tmp/chroma-rag", collection: str = "filings"):
        self._client = chromadb.PersistentClient(path=persist_dir)  # <1>
        self._embed_fn = OpenAIEmbeddingFunction(
            api_key=os.environ["OPENAI_API_KEY"],
            model_name="text-embedding-3-small",
        )
        self._col = self._client.get_or_create_collection(
            name=collection,
            embedding_function=self._embed_fn,
        )

    def add_documents(
        self,
        texts: list[str],
        metadatas: list[dict] | None = None,
        ids: list[str] | None = None,
    ) -> None:
        """Embed and add documents to the collection."""
        n = len(texts)
        ids = ids or [f"doc-{self._col.count() + i}" for i in range(n)]  # <2>
        metadatas = metadatas or [{} for _ in range(n)]
        self._col.add(documents=texts, metadatas=metadatas, ids=ids)

    def query(
        self,
        text: str,
        k: int = 5,
        where: dict | None = None,
    ) -> list[dict]:
        """Retrieve top-k documents, with optional metadata filter."""
        kwargs = {"query_texts": [text], "n_results": min(k, self._col.count())}
        if where:
            kwargs["where"] = where  # <3>
        results = self._col.query(**kwargs)
        docs = results["documents"][0]
        metas = results["metadatas"][0]
        dists = results["distances"][0]
        return [
            {"text": d, "metadata": m, "score": 1 - dist}
            for d, m, dist in zip(docs, metas, dists)
        ]

    def count(self) -> int:
        return self._col.count()

1. `PersistentClient` stores the index on disk, so the collection survives notebook restarts. This matters for production: indexing a large corpus takes minutes; we do not want to re-index on every query.
2. We auto-generate IDs based on the current collection count so that adding documents in multiple batches does not produce ID collisions.
3. The `where` parameter uses ChromaDB's filter syntax: `{"section": {"$eq": "risk_factors"}}`. This translates to a metadata pre-filter before the ANN search — only chunks tagged with the matching metadata are searched.

## Expanded Financial Corpus

We extend the 10-sentence corpus from [notebook 03](/courses/llm-eng/03-rag-concepts.html) with 10 additional passages covering derivatives risk, VaR, Basel III, FINRA compliance, and other topics that require exact-match retrieval to test the limits of dense-only search.

In [ ]:
CORPUS_EXTENDED = [
    # Section: risk_factors (indices 0-4)
    {"text": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point increase in interest rates would reduce the fair value of our fixed-rate debt portfolio by approximately $2.3 billion.", "section": "risk_factors"},
    {"text": "Credit risk arises from the potential that a counterparty will fail to perform its obligations. We manage credit risk through diversification, collateral requirements, and credit limits by counterparty.", "section": "risk_factors"},
    {"text": "Operational risk includes the risk of loss resulting from inadequate or failed internal processes, people, systems, or external events, including cybersecurity threats and technology failures.", "section": "risk_factors"},
    {"text": "Our derivatives portfolio had a notional value of $1.2 trillion at year-end. Net market value exposure after netting and collateral was $18.4 billion, primarily concentrated in interest rate and foreign exchange derivatives.", "section": "risk_factors"},
    {"text": "Our Value at Risk (VaR) at the 99th percentile for a one-day holding period was $142 million, reflecting the diversified nature of our trading portfolios across equities, fixed income, and commodities.", "section": "risk_factors"},

    # Section: mda (indices 5-9)
    {"text": "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year. The increase was driven primarily by higher net interest income reflecting the rising interest rate environment.", "section": "mda"},
    {"text": "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees amid reduced M&A activity and a challenging environment for equity and debt underwriting.", "section": "mda"},
    {"text": "Return on equity for the year was 12.4%, compared to 15.1% in the prior year. Book value per share increased to $312.50, up from $290.20.", "section": "mda"},
    {"text": "Net interest margin expanded 18 basis points to 2.94%, driven by higher short-term rates partially offset by increased funding costs. Provision for credit losses increased to $2.1 billion, reflecting normalization from historically low levels.", "section": "mda"},
    {"text": "Assets under management in our investment management segment grew to $2.8 trillion, an increase of 6% from prior year. Prime brokerage revenues increased 12% to $4.3 billion on higher client balances and margin loan activity.", "section": "mda"},

    # Section: capital_liquidity (indices 10-14)
    {"text": "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.", "section": "capital_liquidity"},
    {"text": "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.", "section": "capital_liquidity"},
    {"text": "Under Basel III framework requirements, our leverage ratio was 5.8%, comfortably above the 3% minimum. Our total risk-weighted assets were $1.47 trillion at year-end, consistent with prior year.", "section": "capital_liquidity"},
    {"text": "We are subject to FINRA Rule 4110 and SEC Rule 15c3-1 (the Net Capital Rule), which require us to maintain minimum net capital of not less than the greater of $250,000 or 2% of aggregate debit items. Our net capital exceeded the minimum by $12.4 billion.", "section": "capital_liquidity"},
    {"text": "The liquidity stress test results indicate the firm could withstand a 30-day severe market stress scenario while maintaining positive liquidity. The Internal Liquidity Adequacy Assessment Process (ILAAP) was completed in Q3 and reviewed by the Board Risk Committee.", "section": "capital_liquidity"},

    # Section: guidance (indices 15-19)
    {"text": "Looking ahead to fiscal 2025, management expects continued revenue growth in the range of 4-6%, supported by a stable rate environment and improving capital markets activity.", "section": "guidance"},
    {"text": "We plan to return $8 billion to shareholders through dividends and share repurchases in fiscal 2025, subject to regulatory approval and market conditions.", "section": "guidance"},
    {"text": "Earnings per share for fiscal 2024 were $42.30, compared to $47.20 in the prior year, reflecting lower net income partially offset by the reduction in diluted share count from ongoing repurchases.", "section": "guidance"},
    {"text": "Management has identified three strategic priorities for fiscal 2025: (1) expanding the wealth management client base to $500 billion in AUM, (2) growing transaction banking revenues by 15%, and (3) reducing the expense ratio below 65%.", "section": "guidance"},
    {"text": "We expect our CET1 ratio to remain in the 13.5–15.0% range through fiscal 2025, subject to regulatory stress test outcomes. Any excess capital above 14.5% will be returned to shareholders under our capital return policy.", "section": "guidance"},
]

print(f"Extended corpus: {len(CORPUS_EXTENDED)} chunks")
for section in ["risk_factors", "mda", "capital_liquidity", "guidance"]:
    count = sum(1 for c in CORPUS_EXTENDED if c["section"] == section)
    print(f"  {section}: {count} chunks")

We index all 20 chunks into ChromaDB with section metadata:

In [ ]:
store = ChromaStore(persist_dir="/tmp/chroma-prep-07", collection="filings-20")

# Clear and reindex (idempotent for notebook re-runs)
try:
    store._client.delete_collection("filings-20")
except Exception:
    pass
store._col = store._client.get_or_create_collection(
    name="filings-20",
    embedding_function=store._embed_fn,
)

texts = [c["text"] for c in CORPUS_EXTENDED]
metas = [{"section": c["section"]} for c in CORPUS_EXTENDED]
ids = [f"chunk-{i:02d}" for i in range(len(texts))]

store.add_documents(texts=texts, metadatas=metas, ids=ids)
print(f"Indexed {store.count()} documents into ChromaDB.")

# Verify metadata filter works
results = store.query("capital adequacy", k=3, where={"section": {"$eq": "capital_liquidity"}})
print(f"\nMetadata-filtered query ('capital_liquidity' section only):")
for r in results:
    print(f"  [{r['score']:.3f}] {r['text'][:80]}...")

## BM25 Retrieval

BM25 scores documents by term frequency — weighted by inverse document frequency — with length normalization. The BM25 Okapi formula is:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot (1 - b + b \cdot |d|/\text{avgdl})}$$

where $k_1 = 1.5$ controls term-frequency saturation and $b = 0.75$ controls length normalization. The key strength for financial retrieval: if a document contains "CET1" exactly, BM25 scores it highly for the query "CET1 ratio" regardless of any semantic embedding. Dense retrieval might rank a semantically similar passage about "capital adequacy" above the passage that literally says "CET1".

In [ ]:
from rank_bm25 import BM25Okapi


class BM25Retriever:
    """BM25 sparse retriever using the rank_bm25 library."""

    def __init__(self, corpus: list[dict]):
        self._corpus = corpus
        tokenized = [doc["text"].lower().split() for doc in corpus]  # <1>
        self._bm25 = BM25Okapi(tokenized)

    def retrieve(self, query: str, k: int = 5) -> list[dict]:
        """Return top-k documents with BM25 scores and zero-based ranks."""
        tokens = query.lower().split()
        scores = self._bm25.get_scores(tokens)  # <2>
        ranked_indices = np.argsort(scores)[::-1][:k]
        return [
            {
                "text": self._corpus[i]["text"],
                "metadata": {k: v for k, v in self._corpus[i].items() if k != "text"},
                "score": float(scores[i]),
                "rank": int(pos),
            }
            for pos, i in enumerate(ranked_indices)
        ]


bm25 = BM25Retriever(CORPUS_EXTENDED)

# Demonstrate exact-match advantage
q_exact = "CET1 ratio regulatory minimum"
q_semantic = "Common Equity Tier 1 capital ratio adequacy"

print(f"BM25 top-2 for exact query: '{q_exact}'")
for r in bm25.retrieve(q_exact, k=2):
    print(f"  [score={r['score']:.2f}] {r['text'][:80]}...")

print(f"\nBM25 top-2 for semantic query: '{q_semantic}'")
for r in bm25.retrieve(q_semantic, k=2):
    print(f"  [score={r['score']:.2f}] {r['text'][:80]}...")

1. BM25 operates on tokenized text. We use simple whitespace tokenization here; in production, use a proper tokenizer that handles punctuation, hyphens in identifiers ("CET1"), and number normalization.
2. `get_scores` returns a score for every document in the corpus in a single vectorized pass — this is efficient even for corpora of 100k documents since BM25 has no quadratic complexity.

## Hybrid Retrieval with RRF

Reciprocal Rank Fusion (RRF) combines ranked lists from multiple retrieval systems without requiring score calibration. Each document $d$ receives a fused score:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

where $R$ is the set of retrieval systems (here: dense + BM25), $\text{rank}_r(d)$ is the 1-based rank of document $d$ in system $r$ (documents not appearing in a list get rank $\infty$, contributing 0 to the sum), and $k = 60$ is a smoothing constant chosen to reduce the influence of top-ranked documents. RRF is robust to score-scale differences: it does not matter that BM25 scores range from 0–5 while cosine similarities range from 0–1.

In [ ]:
def hybrid_retrieve(
    query: str,
    chroma_store: ChromaStore,
    bm25_retriever: BM25Retriever,
    k: int = 5,
    rrf_k: int = 60,
    candidate_k: int = 20,
) -> list[dict]:
    """Hybrid retrieval: dense + BM25 fused with Reciprocal Rank Fusion."""
    # Get top-candidate_k from each retriever
    dense_results = chroma_store.query(query, k=candidate_k)
    bm25_results = bm25_retriever.retrieve(query, k=candidate_k)

    # Build per-document score dicts keyed by text (stable identifier)
    rrf_scores: dict[str, float] = {}
    doc_map: dict[str, dict] = {}

    for rank, doc in enumerate(dense_results, start=1):  # <1>
        key = doc["text"]
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
        doc_map[key] = doc

    for rank, doc in enumerate(bm25_results, start=1):  # <2>
        key = doc["text"]
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
        doc_map[key] = doc

    sorted_keys = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)[:k]
    return [
        {"text": key, "rrf_score": rrf_scores[key], **doc_map[key].get("metadata", {})}
        for key in sorted_keys
    ]

1. Dense results are ranked 1st through `candidate_k`. The 1st result contributes $\frac{1}{61}$ to the RRF score; the 20th contributes $\frac{1}{80}$.
2. BM25 contributions are summed on top: a document that ranks 1st in both retrievers scores approximately $\frac{1}{61} + \frac{1}{61} \approx 0.033$, while a document that only appears in dense at rank 5 scores $\frac{1}{65} \approx 0.015$. The fusion naturally promotes documents that both retrievers agree on.

We build a 10-question test set and measure Recall@3 for dense-only, BM25-only, and hybrid retrieval:

In [ ]:
#| code-fold: true
# Ground truth: (query, ground-truth chunk index in CORPUS_EXTENDED)
RECALL_TEST_SET = [
    ("What is the CET1 capital ratio?", 10),
    ("What is the LCR and HQLA balance?", 11),
    ("How did investment banking revenues perform?", 6),
    ("What is the net interest margin?", 8),
    ("What are the cybersecurity and technology risks?", 2),
    ("What is the Basel III leverage ratio?", 12),
    ("What is the VaR at the 99th percentile?", 4),
    ("FINRA Rule 4110 Net Capital Rule compliance", 13),  # exact-match query
    ("What is the earnings per share?", 17),
    ("What are the assets under management and prime brokerage revenues?", 9),
]


def recall_at_k(results: list[dict], gt_index: int, corpus: list[dict], k: int = 3) -> bool:
    """Check if the ground-truth chunk appears in the top-k results."""
    gt_text = corpus[gt_index]["text"]
    retrieved_texts = [r["text"] for r in results[:k]]
    return gt_text in retrieved_texts


methods = {
    "dense": lambda q: store.query(q, k=3),
    "bm25": lambda q: bm25.retrieve(q, k=3),
    "hybrid": lambda q: hybrid_retrieve(q, store, bm25, k=3),
}

print(f"{'Method':<10} Recall@3")
print("-" * 22)
for method_name, retrieve_fn in methods.items():
    hits = 0
    for query, gt_idx in RECALL_TEST_SET:
        results = retrieve_fn(query)
        if recall_at_k(results, gt_idx, CORPUS_EXTENDED, k=3):
            hits += 1
    print(f"{method_name:<10} {hits}/{len(RECALL_TEST_SET)} = {hits/len(RECALL_TEST_SET):.2f}")

## Cross-Encoder Reranking

A **bi-encoder** (like the OpenAI embedding model) embeds query and document independently and compares them via dot product. This is efficient but asymmetric — the query and document never interact during encoding. A **cross-encoder** concatenates the query and document and runs them through a full transformer together, allowing full self-attention between query and document tokens. Cross-encoders are dramatically more accurate but 100–1000× slower, so we only use them to rerank the top-20 candidates from hybrid retrieval.

In [ ]:
from sentence_transformers import CrossEncoder


class CrossEncoderReranker:
    """Rerank retrieved candidates using a cross-encoder model."""

    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self._model = CrossEncoder(model_name)  # <1>

    def rerank(self, query: str, candidates: list[dict], top_n: int = 5) -> list[dict]:
        """Rescore candidates and return the top_n with updated scores."""
        pairs = [(query, c["text"]) for c in candidates]  # <2>
        scores = self._model.predict(pairs)
        ranked = sorted(
            zip(candidates, scores), key=lambda x: x[1], reverse=True
        )
        return [
            {**doc, "rerank_score": float(score)}
            for doc, score in ranked[:top_n]
        ]


reranker = CrossEncoderReranker()

# Demonstrate: a passage that ranked 8th in hybrid moves to rank 1 after reranking
test_query = "FINRA Rule 4110 Net Capital Rule compliance"
hybrid_top20 = hybrid_retrieve(test_query, store, bm25, k=20)
reranked_top5 = reranker.rerank(test_query, hybrid_top20, top_n=5)

print("Hybrid top-5 (before reranking):")
for i, r in enumerate(hybrid_top20[:5]):
    print(f"  [{i+1}] rrf={r['rrf_score']:.4f} | {r['text'][:70]}...")

print("\nCross-encoder top-5 (after reranking):")
for i, r in enumerate(reranked_top5):
    print(f"  [{i+1}] rerank={r['rerank_score']:.3f} | {r['text'][:70]}...")

1. `cross-encoder/ms-marco-MiniLM-L-6-v2` is a lightweight 22M-parameter cross-encoder trained on the MS MARCO passage ranking dataset. It balances quality and speed for a reranker: ~1ms per (query, passage) pair on CPU.
2. The cross-encoder takes a list of `(query, document)` string pairs and returns a relevance logit for each. Higher is more relevant. The absolute scale of the scores is not meaningful — only their relative ordering matters for reranking.

## RAGAS Evaluation

We define 8 QA pairs with ground-truth answers from the expanded corpus and run `ragas.evaluate()` on three pipeline configurations: dense-only, hybrid, and hybrid + rerank. The goal is to show that each upgrade improves the RAGAS scores.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


def build_ragas_dataset(
    retrieve_fn,
    qa_pairs: list[dict],
    k: int = 3,
) -> Dataset:
    """Build a RAGAS-compatible Dataset by running retrieve_fn for each QA pair."""
    rows = []
    for pair in qa_pairs:
        results = retrieve_fn(pair["question"])
        contexts = [r["text"] for r in results[:k]]
        context_str = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        messages = [
            {"role": "system", "content": "Answer using ONLY the provided context. Cite with [N]."},
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {pair['question']}"},
        ]
        answer = llm.complete(messages)
        rows.append({
            "question": pair["question"],
            "answer": answer,
            "contexts": contexts,
            "ground_truth": pair["ground_truth"],
        })
    return Dataset.from_list(rows)


QA_PAIRS = [
    {"question": "What is the CET1 ratio?", "ground_truth": "14.8% at year-end, above the 4.5% minimum and 13% internal target."},
    {"question": "What is the LCR?", "ground_truth": "128%, exceeding the 100% requirement with $280B in HQLA."},
    {"question": "How did investment banking revenues change?", "ground_truth": "Decreased 23% to $6.1 billion."},
    {"question": "What is the Basel III leverage ratio?", "ground_truth": "5.8%, above the 3% minimum."},
    {"question": "What is the 99th percentile VaR?", "ground_truth": "$142 million for a one-day holding period."},
    {"question": "What is the FINRA Net Capital Rule status?", "ground_truth": "Net capital exceeded the minimum by $12.4 billion under FINRA Rule 4110."},
    {"question": "What is the net interest margin?", "ground_truth": "2.94%, up 18 basis points year-over-year."},
    {"question": "What are AUM and prime brokerage revenues?", "ground_truth": "AUM $2.8 trillion (+6%); prime brokerage revenues $4.3 billion (+12%)."},
]

judge_llm = ChatOpenAI(model="gpt-4o-mini")
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

pipelines = {
    "dense": lambda q: store.query(q, k=20),
    "hybrid": lambda q: hybrid_retrieve(q, store, bm25, k=20),
    "hybrid+rerank": lambda q: reranker.rerank(hybrid_retrieve(q, store, bm25, k=20), query=q, top_n=20),
}

results_table = {}
for name, retrieve_fn in pipelines.items():
    print(f"Evaluating {name}...")
    ds = build_ragas_dataset(retrieve_fn, QA_PAIRS, k=3)
    result = evaluate(
        dataset=ds,
        metrics=[faithfulness, answer_relevancy, context_recall],
        llm=judge_llm,
        embeddings=embed_model,
    )
    results_table[name] = result

print(f"\n{'Pipeline':<15} {'faithfulness':>14} {'ans_relevancy':>14} {'ctx_recall':>12}")
print("-" * 57)
for name, r in results_table.items():
    print(f"{name:<15} {r['faithfulness']:>14.3f} {r['answer_relevancy']:>14.3f} {r['context_recall']:>12.3f}")

## Exercises

1. **Add metadata filtering to restrict retrieval.** Modify `hybrid_retrieve` to accept an optional `section` parameter. When provided, filter the ChromaDB query to `where={"section": {"$eq": section}}` and filter the BM25 results to only include chunks with matching section metadata. Test with `section="capital_liquidity"` on the CET1 query.

2. **Implement a `RetrievalLogger`.** Create a `RetrievalLogger` class that records `{"query": ..., "retrieved_texts": [...], "method": ..., "timestamp": ...}` for each retrieval call. Wrap `hybrid_retrieve` to log every invocation. After running the RAGAS evaluation, print a summary of which chunks were retrieved most frequently.

3. **Tune the RRF $k$ parameter.** Write a loop over `rrf_k in [10, 30, 60, 120]`. For each value, compute Recall@3 on the 10-question test set from the hybrid retrieval section. Plot (or print) how Recall@3 varies with $k$ and explain why the curve is non-monotone.

---

$\blacksquare$